# Notebook 3: 03_Live_Clustering

This notebook simulates real-time data ingestion. It filters the dataset for a specific 60-minute 'live' window, runs spatial HDBSCAN clustering on those active violations, and calculates the Live Density ($D_{live}$) component of the Hotspot Risk Score.

### Cell 1: Setup & Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import hdbscan
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Loading Feature-Rich Foundation Data...")

# Load the parquet file generated in Notebook 2
FEATURE_DATA_PATH = "data/processed/hotspot_features.parquet"
try:
    df = pd.read_parquet(FEATURE_DATA_PATH)
    print(f"✅ Successfully loaded {len(df)} records.")
except FileNotFoundError:
    print("❌ Error: hotspot_features.parquet not found. Please run Notebook 2 first.")

Loading Feature-Rich Foundation Data...
✅ Successfully loaded 184411 records.


### Cell 2: The 'Live Feed' Time-Window Simulator
To test our live clustering, we will simulate a high-traffic moment. From our Notebook 1 analysis, we know 10:30 AM is peak morning rush.

In [ ]:
print("Simulating Live Data Stream...")

# 1. Define the 'Live' Moment (We use a known busy day/time from the dataset)
SIMULATED_NOW = pd.to_datetime("2023-11-28 10:30:00").tz_localize("Asia/Kolkata")
TIME_WINDOW_MINS = 60

# 2. Calculate the cutoff time
cutoff_time = SIMULATED_NOW - pd.Timedelta(minutes=TIME_WINDOW_MINS)

# 3. Filter the dataset to represent what the cameras see 'Right Now'
live_df = df[(df["created_ist"] <= SIMULATED_NOW) & (df["created_ist"] >= cutoff_time)].copy()

print(f"✅ Live simulation set to: {SIMULATED_NOW}")
print(f"🚨 Active violations in the last {TIME_WINDOW_MINS} minutes: {len(live_df)}")

Simulating Live Data Stream...
✅ Live simulation set to: 2023-11-28 10:30:00+05:30
🚨 Active violations in the last 60 minutes: 125


### Cell 3: Live Spatio-Temporal Clustering (HDBSCAN)
Because we already filtered the data to a tight 60-minute window, we only need to cluster spatially. Any cluster found here is, by definition, a concurrent traffic jam.

In [ ]:
print("Executing Real-Time HDBSCAN...")

# 1. Extract coordinates and convert to radians for Haversine distance
coords = np.radians(live_df[["latitude", "longitude"]].dropna().values)

# 2. Configure HDBSCAN for live detection
# We drop the min_cluster_size to 5. In a 60 min window, 5 illegally parked cars in one spot is an active threat.
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    metric="haversine",
    core_dist_n_jobs=-1 # Uses all CPU cores for faster execution
)
live_df["live_cluster_id"] = clusterer.fit_predict(coords)

# 3. Filter out noise (Cluster -1)
active_hotspots = live_df[live_df["live_cluster_id"] != -1].copy()

total_live_clusters = active_hotspots["live_cluster_id"].nunique()
print(f"✅ Detected {total_live_clusters} critical live clusters forming right now.")

Executing Real-Time HDBSCAN...
✅ Detected 11 critical live clusters forming right now.


### Cell 4: Engineering the Live Density Score ($D_{live}$)
We calculate the immediate severity of each live cluster by summing the impact of the vehicles in it, then normalizing it to a 0-100 score.

In [ ]:
print("Calculating Live Density Scores...")

# Aggregate stats for each active cluster
live_stats = active_hotspots.groupby("live_cluster_id").agg(
    active_vehicles=("device_id", "count"),
    total_live_impact=("base_impact_score", "sum"),
    center_lat=("latitude", "mean"),
    center_lon=("longitude", "mean")
).reset_index()

# Map these cluster centers back to our master 100x100m grid system
live_stats["grid_lat"] = live_stats["center_lat"].round(3)
live_stats["grid_lon"] = live_stats["center_lon"].round(3)

# Calculate D_live (Live Density Score)
# We cap the theoretical max impact at 500 (approx. 20-30 heavy vehicles).
# Anything over 500 is an immediate 100/100 crisis.
MAX_EXPECTED_IMPACT = 500
live_stats["live_density_score"] = (live_stats["total_live_impact"] / MAX_EXPECTED_IMPACT) * 100

# Cap the score at 100
live_stats["live_density_score"] = live_stats["live_density_score"].clip(upper=100)

print("✅ Live Density ($D_{live}$) generated.")
display(live_stats.sort_values("live_density_score", ascending=False).head(5))

Calculating Live Density Scores...
✅ Live Density ($D_{live}$) generated.


,live_cluster_id,active_vehicles,total_live_impact,center_lat,center_lon,grid_lat,grid_lon,live_density_score
10,10,11,360.0,12.977124,77.576614,12.977,77.577,72.0
6,6,10,330.0,12.977992,77.604612,12.978,77.605,66.0
5,5,12,330.0,12.974421,77.543687,12.974,77.544,66.0
9,9,8,250.0,12.976145,77.579254,12.976,77.579,50.0
4,4,10,220.0,13.009288,77.550905,13.009,77.551,44.0


### Cell 5: Exporting the Live Reference State
In the final Streamlit app, we will run this dynamically whenever the user moves the time slider. Here, we export a snapshot to validate the logic works before integrating it.

In [ ]:
print("Exporting Live Reference Artifacts...")

# Keep only the essential columns for the live dashboard state
live_export = live_stats[[
    "grid_lat", "grid_lon",
    "active_vehicles", "live_density_score"
]]

LIVE_EXPORT_PATH = "data/processed/live_reference.csv"
live_export.to_csv(LIVE_EXPORT_PATH, index=False)

print(f"🚀 SUCCESS: Live pipeline simulation complete. Snapshot saved to {LIVE_EXPORT_PATH}")

Exporting Live Reference Artifacts...
🚀 SUCCESS: Live pipeline simulation complete. Snapshot saved to data/processed/live_reference.csv
